In [4]:
# !pip install hurst
# !pip install arch 
# !pip install lightgbm 
# !pip install scikit-learn 
# !pip install pmdarima
# !pip install shap
# !pip install numpy pandas
# Fed API: 1f3befec93b6c68383d27f3b58cca038
# !pip install fredapi
# 	Final_Historical_Score
# CNY	0.74. 1
# INR	0.69. 2
# AUD	0.57
# CAD	0.54. 10
# CHF	0.52
# USD	0.48. 6
# JPY	0.46. 4
# GBP	0.42. 5
# EUR	0.38. 9. Ger is 3. Ital is 8. Fra is 9

In [2]:
# Import the libraries
import pyautogui
import time
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import os
import itertools

from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)  # or None for no limit

In [3]:
#To run the modules created
import sys
sys.path.append("..")                 # if needed to find project_pkg
import project_pkg.dld as dld  # import module object
import project_pkg.dld_h4 as dh4 # import module object
from importlib import reload
reload(dld)
reload(dh4)

# call function/class inside the module
Curr2 = [#Primary: 
         'USDJPY', 'EURUSD','US30.a','XAUUSD.a','NAS100.a','USDINR'
         #Secondary:
        ,'USTN10YR-F.a','SpotBrent','VIX.a','USDX.a']
# XAU: 17bar 2 o/c .1766 means -0.0695 bar 
# USDJPY: ±0.3903
filtered_df = pd.DataFrame({'Currency': ['AUDUSD', *Curr2]})

dld.dld(Curr2)
dh4.dld(Curr2)

import project_pkg.utils as utils

In [4]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

# 1. Added run_flavor_stress_tests to the import list
from project_pkg.model_h4 import (
    calculate_technical_indicators,
    run_block_walk_forward_validation,  
    run_flavor_stress_tests,         # <--- Restored import
    predict_next_day_scenarios,
    MEAN_REVERSION_FEATURES,
    PURE_MOMENTUM_FEATURES
)


# Silence diagnostic warnings
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
warnings.filterwarnings("ignore", message=".*No supported index is available.*")
    
# Process loop
for currency in filtered_df['Currency']:
    if pd.isna(currency):
        continue
        
    file_path = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}.csv"
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    
    print(f" INITIALIZING PARALLEL DUAL-STRATEGY PIPELINE FOR: {currency} ")
    print("=" * 115)
    
    try:
        df_mod = pd.read_csv(file_path, sep='\t', engine='python')
        df_mod_h4 = pd.read_csv(file_path_h4, sep='\t', engine='python')
    except Exception as e:
        print(f"Skipping {currency}: Failed to load file data. Error: {e}")
        continue

    # Guard clause for dataset window boundaries
    if len(df_mod) < 800:
        print(f"Skipping {currency}: Historical dataset size ({len(df_mod)} rows) insufficient.")
        continue

    # --- 1. RUN CORRECTED BASELINE WALK-FORWARD BACKTEST ---
    print("Running historical backtest cross-validations (Purged & Embargoed)...")
    mae_rev, mae_mom = run_block_walk_forward_validation(
        df_daily=df_mod,
        df_h4=df_mod_h4, 
        train_window=700,
        retrain_interval=100
    )
    print(f" Backtest Metrics Summary -> Engine 1 (Reversion) MAE: {mae_rev:.5f} | Engine 2 (Momentum) MAE: {mae_mom:.5f}")

    # --- 2. RUN HISTORICAL MACRO FLAVOR STRESS TESTS ---
    print("\nInjecting synthetic macro disruptions and computing shock resiliency...")
    run_flavor_stress_tests(df_mod, df_mod_h4, train_window=700) # <--- Triggers Flavor Matrix

    # --- 3. PRINT THEMATIC FEATURE MAP REFERENCE ---
#     print(f"[ACTIVE STRATEGY FEATURE MAP FOR {currency}]:")
#     print("-" * 115)
#     print(f"{'Engine Matrix Allocation':<25} | {'Monitored Feature':<25} | {'Functional Impact Description'}")
#     print("-" * 115)
#     for feat, desc in list(MEAN_REVERSION_FEATURES.items())[:3]:
#         print(f"{'1. Mean Reversion':<25} | {feat:<25} | {desc}")
#     for feat, desc in list(PURE_MOMENTUM_FEATURES.items())[:3]:
#         print(f"{'2. Pure Momentum':<25} | {feat:<25} | {desc}")
#     print("-" * 115)

    # --- 4. GENERATE MULTI-ENGINE SCENARIOS & RE-SCALED TARGETS ---
    print(f"Generating tomorrow's expected returns and nominal pricing scenarios...")
    predict_next_day_scenarios(df_mod, df_mod_h4, train_window=700)

 INITIALIZING PARALLEL DUAL-STRATEGY PIPELINE FOR: AUDUSD 
Running historical backtest cross-validations (Purged & Embargoed)...
 Backtest Metrics Summary -> Engine 1 (Reversion) MAE: 0.00457 | Engine 2 (Momentum) MAE: 0.00453

Injecting synthetic macro disruptions and computing shock resiliency...

-----------------------------------------------------------------
 TAIL RISK STRESS TEST RESULTS MATRIX (HISTORICAL MACRO FLAVORS)
-----------------------------------------------------------------
 MACROECONOMIC SIMULATED CRITICAL FLAVOR | ENG1 MAE  | ENG2 MAE
-----------------------------------------------------------------
 Flavor 1 (Liquidity Black Hole)  | 0.00450   | 0.00446
 Flavor 2 (Structural Bear Mkt)   | 0.01413   | 0.01392
 Flavor 3 (High-Vol Grind)        | 0.01690   | 0.01749
 Flavor 4 (The Gold Rush Bubble)  | 0.18767   | 3.72990
-----------------------------------------------------------------

Generating tomorrow's expected returns and nominal pricing scenarios...

        

 Backtest Metrics Summary -> Engine 1 (Reversion) MAE: 0.00358 | Engine 2 (Momentum) MAE: 0.00359

Injecting synthetic macro disruptions and computing shock resiliency...

-----------------------------------------------------------------
 TAIL RISK STRESS TEST RESULTS MATRIX (HISTORICAL MACRO FLAVORS)
-----------------------------------------------------------------
 MACROECONOMIC SIMULATED CRITICAL FLAVOR | ENG1 MAE  | ENG2 MAE
-----------------------------------------------------------------
 Flavor 1 (Liquidity Black Hole)  | 0.00359   | 0.00359
 Flavor 2 (Structural Bear Mkt)   | 0.01544   | 0.01557
 Flavor 3 (High-Vol Grind)        | 0.01685   | 0.01645
 Flavor 4 (The Gold Rush Bubble)  | 0.35683   | 2.53635
-----------------------------------------------------------------

Generating tomorrow's expected returns and nominal pricing scenarios...

                                           WEEKLY MACRO HEALTH CHECK MATRIX                                             
 [STRUCTURAL TRE

 Backtest Metrics Summary -> Engine 1 (Reversion) MAE: 0.01083 | Engine 2 (Momentum) MAE: 0.01094

Injecting synthetic macro disruptions and computing shock resiliency...

-----------------------------------------------------------------
 TAIL RISK STRESS TEST RESULTS MATRIX (HISTORICAL MACRO FLAVORS)
-----------------------------------------------------------------
 MACROECONOMIC SIMULATED CRITICAL FLAVOR | ENG1 MAE  | ENG2 MAE
-----------------------------------------------------------------
 Flavor 1 (Liquidity Black Hole)  | 0.01097   | 0.01094
 Flavor 2 (Structural Bear Mkt)   | 0.01482   | 0.01481
 Flavor 3 (High-Vol Grind)        | 0.01696   | 0.01718
 Flavor 4 (The Gold Rush Bubble)  | 0.29058   | 3.79632
-----------------------------------------------------------------

Generating tomorrow's expected returns and nominal pricing scenarios...

                                           WEEKLY MACRO HEALTH CHECK MATRIX                                             
 [STRUCTURAL TRE